In [ ]:
# --- SYSTEM & ENVIRONMENT ---
import os
import time
import torch
import pandas as pd
from dotenv import load_dotenv

# --- LANGCHAIN & VECTOR DB ---
import chromadb
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma  # Updated from community to langchain_chroma
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_experimental.text_splitter import SemanticChunker


# --- INITIALIZATION ---
load_dotenv(override=True)

# Verify CUDA for your powerful system
cuda_available = torch.cuda.is_available()
print(f"✅ CUDA Available: {cuda_available}")
if cuda_available:
    print(f"🚀 Using GPU: {torch.cuda.get_device_name(0)}")


# Initializing HF Embeddings on your NVIDIA GPU
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={'device': 'cuda' if cuda_available else 'cpu'}
)

# --- 2. VECTOR DB PERSISTENCE ---
persist_directory = r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\VectorDatabase"

# Initialize Client to ensure the folder exists and is managed correctly
client = chromadb.PersistentClient(path=persist_directory)

# Connect LangChain to the existing collection
vectorstore = Chroma(
    client=client,
    collection_name="legal_knowledge",
    embedding_function=embeddings
)

print(f"✅ Vector Database Connected. Chunks: {vectorstore._collection.count()}")

✅ CUDA Available: True
🚀 Using GPU: NVIDIA GeForce RTX 4060 Ti


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Vector Database Connected. Chunks: 898


In [2]:
from langchain_community.document_loaders import PyMuPDFLoader

# 1. Update File Path: Point to your specific PDF
file_path = r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\data\Indian_Constitution.pdf"

# 2. Load the PDF using LangChain's specialized loader
# PyMuPDF is excellent for maintaining the layout of legal documents
if os.path.exists(file_path):
    loader = PyMuPDFLoader(file_path)
    # vector_docs will be a list of LangChain Document objects
    vector_docs = loader.load()
    print(f"✅ Successfully loaded {len(vector_docs)} pages from the Indian Constitution.")
else:
    print(f"❌ Error: The file at {file_path} does not exist.")

✅ Successfully loaded 402 pages from the Indian Constitution.


In [3]:
print(vector_docs[0])

page_content='£ÉÉ®iÉ BÉEÉ ºÉÆÉÊ´ÉvÉÉxÉ 
[1
, 2024
]
THE CONSTITUTION OF INDIA
[As on 1st May, 2024] 
2024
GOVERNMENT OF INDIA
MINISTRY OF LAW AND JUSTICE
LEGISLATIVE DEPARTMENT, OFFICIAL LANGUAGES WING' metadata={'producer': 'iLovePDF', 'creator': '', 'creationdate': '', 'source': 'C:\\Users\\user\\Desktop\\RAG Projects\\Legal RAG 1\\data\\Indian_Constitution.pdf', 'file_path': 'C:\\Users\\user\\Desktop\\RAG Projects\\Legal RAG 1\\data\\Indian_Constitution.pdf', 'total_pages': 402, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-07-01T11:20:33+00:00', 'trapped': '', 'modDate': 'D:20240701112033Z', 'creationDate': '', 'page': 0}


In [4]:
# articles short
#Pages 3 to 31 contains the  introduction
articles_short = vector_docs[3:31]
print(articles_short[0])

page_content='THE CONSTITUTION OF INDIA
____________                                                                    
CONTENTS
__________
                                                                                          
PREAMBLE
PART I 
THE UNION AND ITS TERRITORY
ARTICLES 
1.
Name and territory of the Union.
  2. 
Admission or establishment of new States. 
[2A.         Sikkim to be associated with the Union.—Omitted.]
3.
Formation of new States and alteration of areas, boundaries or 
names of existing States.
4.
Laws made under articles 2 and 3 to provide for the amendment of 
the First and the Fourth Schedules and supplemental, incidental 
and consequential matters.
PART II
CITIZENSHIP
5.
Citizenship at the commencement of the Constitution.
6.
Rights of citizenship of certain persons who have migrated to 
India from Pakistan.
7.
Rights of citizenship of certain migrants to Pakistan.
8.
Rights of citizenship of certain persons of Indian origin residing    
outside India.
9

In [5]:
# preamble
preamble = vector_docs[31]
print(preamble)

page_content='THE CONSTITUTION OF INDIA 
PREAMBLE
WE, THE PEOPLE OF INDIA, having solemnly resolved to constitute 
India into a 
1[SOVEREIGN SOCIALIST SECULAR DEMOCRATIC 
REPUBLIC] and to secure to all its citizens:
JUSTICE, social, economic and political;
 
LIBERTY of thought, expression, belief, faith and worship;
EQUALITY of status and of opportunity;
and to promote among them all
FRATERNITY assuring the dignity of the individual and the 2[unity 
and integrity of the Nation];
IN OUR CONSTITUENT ASSEMBLY this twenty-sixth day of 
November, 1949, do HEREBY ADOPT, ENACT AND GIVE TO 
OURSELVES THIS CONSTITUTION.
______________________________________________
1. Subs. by the Constitution (Forty-second Amendment) Act, 1976, s.2, for "SOVEREIGN 
DEMOCRATIC REPUBLIC" (w.e.f. 3-1-1977).
2. Subs. by s. 2, ibid., for "Unity of the Nation" (w.e.f. 3-1-1977).' metadata={'producer': 'iLovePDF', 'creator': '', 'creationdate': '', 'source': 'C:\\Users\\user\\Desktop\\RAG Projects\\Legal RAG 1\\data

In [6]:
articles = vector_docs[32:283]

In [7]:
print(articles[0].page_content)

2
PART I
THE UNION AND ITS TERRITORY
1. Name and territory of the Union.—(1) India, that is Bharat, 
shall be a Union of States.
1[(2) The States and the territories thereof shall be as specified in 
the First Schedule.]
(3) The territory of India shall comprise—
(a) the territories of the States; 
2[(b) the Union territories specified in the First Schedule; 
and]
(c) such other territories as may be acquired.
2. Admission or establishment of new States.—Parliament may 
by law admit into the Union, or establish, new States on such terms and 
conditions as it thinks fit.
3[2A. [Sikkim to be associated with the Union.] —Omitted by the 
Constitution (Thirty-sixth Amendment) Act, 1975, s. 5 (w.e.f. 26-4-1975).]
3. Formation of new States and alteration of areas, 
boundaries or names of existing States.—Parliament may by law—
(a) form a new State by separation of territory from any 
State or by uniting two or more States or parts of States or by 
uniting any territory to a part of any State

In [8]:
##code to remove headers  and footers from pdf for vector db
import re
for art in articles:
  art.page_content = re.split(r'_{2,}', art.page_content)[0].strip()

In [9]:
import re

# 1. Combine the raw LangChain objects
# Ensure all source lists are present and correctly ordered
all_vector_docs = articles_short + [preamble] + articles

# 2. Advanced Metadata Alignment & Text Cleaning
# This block ensures 'page_label' is correctly mapped from the loader's metadata
for doc in all_vector_docs:
    # A. Metadata Synchronization
    # PyMuPDF stores the page number in 'page' (0-indexed)
    # We retrieve it and convert it to a 1-indexed string for legal reference
    raw_page_num = doc.metadata.get("page")
    
    if raw_page_num is not None:
        # Convert 0-indexed to 1-indexed (Page 0 -> Page 1)
        doc.metadata["page_label"] = str(int(raw_page_num) + 1)
    else:
        # Fallback if the loader missed a page
        doc.metadata["page_label"] = "Unknown"

    # B. Text Cleaning (Post-Processing)
    # Access raw text via LangChain's page_content attribute
    raw_text = doc.page_content
    
    # Apply your specific legal cleaning:
    # 1. Remove non-ASCII characters (Hindi/Diglot noise) to focus on English law
    cleaned_ascii = "".join(i for i in raw_text if ord(i) < 128)
    
    # 2. Split by underscores (footer noise) and strip whitespace
    cleaned_final = re.split(r'_{2,}', cleaned_ascii)[0].strip()
    
    # Update the document with cleaned text
    doc.page_content = cleaned_final

print(f"✅ Successfully consolidated {len(all_vector_docs)} documents.")
print(f"✅ Metadata Verified: 'page_label' mapped for Small-to-Big retrieval.")

✅ Successfully consolidated 280 documents.
✅ Metadata Verified: 'page_label' mapped for Small-to-Big retrieval.


In [10]:
import re

# --- STEP 1: LOAD AND TAG SCHEDULES ---

# 1. Slice the schedules from your original vector_docs (LangChain objects)
# Note: Ensure vector_docs is the list generated by PyMuPDFLoader in Step 1
schedules_docs = vector_docs[283:381]

# 2. Advanced Metadata Alignment for Schedules
# This ensures 'page_label' is a 1-indexed string to avoid 'Unknown' errors
for d in schedules_docs:
    raw_page_num = d.metadata.get("page")
    
    if raw_page_num is not None:
        # Convert 0-indexed integer to 1-indexed string (e.g., 283 -> "284")
        actual_page = str(int(raw_page_num) + 1)
    else:
        actual_page = "Unknown"
        
    d.metadata.update({
        "page_label": actual_page,
        "section_type": "Schedule",
        "document_part": "Schedules of the Constitution"
    })

# --- STEP 2: UNIFIED CLEANING (Non-ASCII & Underscores) ---

for doc in schedules_docs:
    # 1. Access raw text via LangChain attribute
    raw_text = doc.page_content
    
    # 2. Remove Non-ASCII characters
    # Removes Hindi 'Diglot' noise to optimize for BGE English embeddings on CUDA
    cleaned = "".join(i for i in raw_text if ord(i) < 128)
    
    # 3. Remove footer underscores (common in official document layouts)
    cleaned = re.split(r'_{2,}', cleaned)[0].strip()
    
    # 4. Update the LangChain document content
    doc.page_content = cleaned

# --- STEP 3: MERGE INTO YOUR FINAL LIST ---

# all_vector_docs now contains: Preamble + Articles + Schedules
# This consolidated list will be passed to the Semantic Chunker
all_vector_docs = all_vector_docs + schedules_docs

print(f"✅ Schedules processed and merged. Total Document Count: {len(all_vector_docs)}")
print(f"📍 Sample Metadata from last schedule: {all_vector_docs[-1].metadata}")

✅ Schedules processed and merged. Total Document Count: 378
📍 Sample Metadata from last schedule: {'producer': 'iLovePDF', 'creator': '', 'creationdate': '', 'source': 'C:\\Users\\user\\Desktop\\RAG Projects\\Legal RAG 1\\data\\Indian_Constitution.pdf', 'file_path': 'C:\\Users\\user\\Desktop\\RAG Projects\\Legal RAG 1\\data\\Indian_Constitution.pdf', 'total_pages': 402, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-07-01T11:20:33+00:00', 'trapped': '', 'modDate': 'D:20240701112033Z', 'creationDate': '', 'page': 380, 'page_label': '381', 'section_type': 'Schedule', 'document_part': 'Schedules of the Constitution'}


In [11]:
# Step 5: Robust Semantic Chunking
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings, 
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=90
)

print("🚀 Splitting and mapping metadata on GPU...")

# We map metadata manually after splitting to ensure NO data loss
semantic_chunks = []
for original_doc in all_vector_docs:
    # Split each page individually to keep metadata localized
    chunks_from_page = semantic_chunker.split_documents([original_doc])
    
    for chunk in chunks_from_page:
        # Explicitly force metadata inheritance from the original page
        chunk.metadata["page_label"] = original_doc.metadata.get("page_label", "Unknown")
        chunk.metadata["section_type"] = original_doc.metadata.get("section_type", "Article")
        
        # Implement the "Small-to-Big" Context
        # We store the FULL original page text as the 'parent_context'
        chunk.metadata["parent_context"] = original_doc.page_content
        
        semantic_chunks.append(chunk)

print(f"✅ Created {len(semantic_chunks)} chunks with guaranteed Metadata preservation.")

🚀 Splitting and mapping metadata on GPU...


✅ Created 898 chunks with guaranteed Metadata preservation.


In [12]:
def inspect_semantic_chunks(chunks, name="Chunks"):
    print(f"--- 🔍 {name} Inspection Report ---")
    print(f"Total Semantic Chunks Created: {len(chunks)}\n")
    
    # Check the first 3 chunks to verify cleaning and semantic breaks
    for i, chunk in enumerate(chunks[:3]):
        # Metadata check for 'page_label' and 'section_type' (Articles vs Schedules)
        page = chunk.metadata.get('page_label', 'Unknown')
        section = chunk.metadata.get('section_type', 'General/Article')
        
        print(f"CHUNK {i} | Page: {page} | Type: {section}")
        
        # In LangChain, we access text via .page_content
        text = chunk.page_content.replace('\n', ' ')
        
        # Previewing the semantic 'boundaries'
        print(f"START: {text[:200]}...")
        print(f"END:   ...{text[-200:]}")
        
        # Verify if Parent Context is being captured for Small-to-Big RAG
        if "parent_context" in chunk.metadata:
            print("✅ Context Status: Parent Context Attached")
            
        print("-" * 50)

# 2. Inspect your semantic chunks
inspect_semantic_chunks(semantic_chunks, "Legal Vector Store")

--- 🔍 Legal Vector Store Inspection Report ---
Total Semantic Chunks Created: 898

CHUNK 0 | Page: 4 | Type: Article
START: THE CONSTITUTION OF INDIA...
END:   ...THE CONSTITUTION OF INDIA
✅ Context Status: Parent Context Attached
--------------------------------------------------
CHUNK 1 | Page: 5 | Type: Article
START: Contents ARTICLES (ii) PART III FUNDAMENTAL RIGHTS General  12. Definition....
END:   ...Contents ARTICLES (ii) PART III FUNDAMENTAL RIGHTS General  12. Definition.
✅ Context Status: Parent Context Attached
--------------------------------------------------
CHUNK 2 | Page: 5 | Type: Article
START: 13. Laws inconsistent with or in derogation of the fundamental  rights. Right to Equality 14. Equality before law....
END:   ...13. Laws inconsistent with or in derogation of the fundamental  rights. Right to Equality 14. Equality before law.
✅ Context Status: Parent Context Attached
--------------------------------------------------


## Vector Store

In [13]:
# Use the persistent client we initialized in Step 1
# This ensures your 894 chunks are saved to your local folder
vector_store = Chroma.from_documents(
    documents=semantic_chunks,
    embedding=embeddings,
    client=client,
    collection_name="legal_knowledge"
)
print(f"✅ Successfully persisted {vector_store._collection.count()} chunks to disk.")

✅ Successfully persisted 898 chunks to disk.
